In [8]:
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from itertools import combinations
import numpy as np
# Данные

data = pd.DataFrame({
    'Age_Group': [1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5],
    'Ig_A': [83, 85, 84, 85, 85, 86, 86, 87, 86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90, 89, 90, 90, 91, 90, 92]
})

group_data = {
    1: [83, 85],
    2: [84, 85, 85, 86, 86, 87],
    3: [86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90],
    4: [89, 90, 90, 91],
    5: [90, 92]
}

alpha = 0.05

ПУНКТ а) Влияет ли возраст на содержание Ig A

In [10]:
# Базовая группа (референсная) — группа 3
base_group = 3

# Создаём индикаторные переменные
dummies = pd.get_dummies(data['Age_Group'], prefix='Age', dtype=int)
dummies = dummies.drop(columns=[f'Age_{base_group}'])

# 2a. Проверка на мультиколлинеарность
for col in dummies.columns:
    X_other = dummies.drop(columns=[col])
    X_other_const = sm.add_constant(X_other)
    model_aux = sm.OLS(dummies[col], X_other_const).fit()
    r2 = model_aux.rsquared
    status = "проверь" if r2 > 0.7 else "OK"
    print(f"  {col}: R² = {r2:.4f} {status}")

# 2b. Построение регрессии
X = sm.add_constant(dummies)
model = sm.OLS(data['Ig_A'], X).fit()

print("\n--- Результаты регрессии ---")
print(f"R² модели = {model.rsquared:.5f}")
print("\nКоэффициенты:")
for param, value, pval in zip(model.params.index, model.params.values, model.pvalues):
    print(f"  {param}: {value:.4f} (p-value = {pval:.6f})")

# 3. дисперсионный анализ

anova_model = ols('Ig_A ~ C(Age_Group)', data=data).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=2)

print(anova_table)

p_value = anova_table.loc['C(Age_Group)', 'PR(>F)']

if p_value < 0.05:
    print("\nПрисутсвует значимое различие между группами (p-value < 0.05)")
else:
    print("\nНе присутсвует значимое различие между группами")


  Age_1: R² = 0.0803 OK
  Age_2: R² = 0.1486 OK
  Age_4: R² = 0.1270 OK
  Age_5: R² = 0.0803 OK

--- Результаты регрессии ---
R² модели = 0.81061

Коэффициенты:
  const: 87.8182 (p-value = 0.000000)
  Age_1: -3.8182 (p-value = 0.000166)
  Age_2: -2.3182 (p-value = 0.000395)
  Age_4: 2.1818 (p-value = 0.002393)
  Age_5: 3.1818 (p-value = 0.001003)
                 sum_sq    df     F        PR(>F)
C(Age_Group)  99.023636   4.0  21.4  5.407435e-07
Residual      23.136364  20.0   NaN           NaN

Присутсвует значимое различие между группами (p-value < 0.05)


ПУНКТ б) Попарно сравнить средние в каждой группе

In [17]:
pairs = [(i, j) for i in range(1, 6) for j in range(i + 1, 6)]

# 4a. Проверка равенства дисперсий
print("--- Проверка равенства дисперсий (F-тест) ---")

for g1, g2 in pairs:
    data1 = group_data[g1]
    data2 = group_data[g2]
    n1, n2 = len(data1), len(data2)
    
    var1 = np.var(data1, ddof=1)
    var2 = np.var(data2, ddof=1)
    
    if var1 >= var2:
        F_val = var1 / var2
        df1, df2 = n1 - 1, n2 - 1
    else:
        F_val = var2 / var1
        df1, df2 = n2 - 1, n1 - 1
    
    p_value = stats.f.sf(F_val, df1, df2)
    result = "равны" if p_value > 0.05 else "не равны"
    print(f"  Группы {g1}-{g2}: F = {F_val:.4f}, p-value = {p_value:.4f} → {result}")

# 4b. T-тест (предполагаем равные дисперсии, т.к. выше все p-value > 0.05)
print("\n--- T-тест (равные дисперсии) ---")

ttest_results = []
for g1, g2 in pairs:
    t_stat, p_value = stats.ttest_ind(group_data[g1], group_data[g2], equal_var=True)
    result = "значимо" if p_value < 0.05 else "не значимо"
    print(f"  Группы {g1}-{g2}: t = {t_stat:.4f}, p-value = {p_value:.6f} → {result}")
    ttest_results.append((g1, g2, p_value))

# 4c. Поправка Бонферрони
print("\n=== Поправка Бонферрони ===")

m = len(pairs)  # 10 сравнений
alpha_bonferroni = 0.05 / m

print(f"Уровень значимости с поправкой Бонферрони: α = 0.05/{m} = {alpha_bonferroni:.6f}\n")

significant_count = 0
for g1, g2, p_val in ttest_results:
    if p_val < alpha_bonferroni:
        significant_count += 1
        print(f"  Группы {g1}-{g2}: p-value = {p_val:.6f} < {alpha_bonferroni:.6f} → ЗНАЧИМОЕ различие")
    else:
        print(f"  Группы {g1}-{g2}: p-value = {p_val:.6f} ≥ {alpha_bonferroni:.6f} → НЕ значимо")

print(f"\nИтого значимых различий после поправки Бонферрони: {significant_count} из {m}")

--- Проверка равенства дисперсий (F-тест) ---
  Группы 1-2: F = 1.8182, p-value = 0.2354 → равны
  Группы 1-3: F = 1.7188, p-value = 0.2192 → равны
  Группы 1-4: F = 3.0000, p-value = 0.1817 → равны
  Группы 1-5: F = 1.0000, p-value = 0.5000 → равны
  Группы 2-3: F = 1.0579, p-value = 0.5070 → равны
  Группы 2-4: F = 1.6500, p-value = 0.3609 → равны
  Группы 2-5: F = 1.8182, p-value = 0.2354 → равны
  Группы 3-4: F = 1.7455, p-value = 0.3544 → равны
  Группы 3-5: F = 1.7188, p-value = 0.2192 → равны
  Группы 4-5: F = 3.0000, p-value = 0.1817 → равны

--- T-тест (равные дисперсии) ---
  Группы 1-2: t = -1.6432, p-value = 0.151454 → не значимо
  Группы 1-3: t = -4.4611, p-value = 0.000961 → значимо
  Группы 1-4: t = -6.9282, p-value = 0.002278 → значимо
  Группы 1-5: t = -4.9497, p-value = 0.038476 → значимо
  Группы 2-3: t = -4.2735, p-value = 0.000666 → значимо
  Группы 2-4: t = -7.2000, p-value = 0.000092 → значимо
  Группы 2-5: t = -6.0249, p-value = 0.000944 → значимо
  Группы 3-4: 